# 前処理の考え方
機械学習のモデルにデータを入れる前に必要な「前処理」の目的と、守るべきルールを確認します。学習用と予測用のデータに同じ処理をすること、補完などに使う値は学習用のデータだけから求めることを理解します。

## 学習目標
- モデルに入れるデータが満たすべき条件(数値であること、欠損がないこと)を説明できる
- 学習用と予測用のデータに同じ前処理をする必要がある理由を説明できる
- 補完などに使う値を学習用のデータだけから求める理由(リークの防止)を説明できる
- 予測に使う列と使わない列を選び分けられる

## なぜ前処理が必要か

前の講座(データの理解)では、性別・等級・年齢などが生存と関係していることを確かめました。次の講座では、これらの情報から生存を予測する **機械学習のモデル** を作ります。

多くのモデル(scikit-learn のモデルなど)には、入力のデータについて次の条件があります。

| 条件 | 今のデータの状態 | 必要な前処理 |
|---|---|---|
| すべての値が **数値** である | `Sex`(male / female)・`Embarked`(C / Q / S)などが文字列 | カテゴリ変数の数値化(単元3) |
| **欠損値がない** | `Age`・`Embarked`・`Fare` に欠損がある | 欠損値の補完(単元2) |
| 予測に役立つ情報が入っている | 名前・家族の人数などの情報が、そのままでは使いにくい | 特徴量の作成(単元4) |

このような、データをモデルに入れられる形に整える作業を **前処理**、モデルに入れる1つ1つの列を **特徴量** といいます。

## 予測に使う列を選ぶ

まず、データの列を「予測に使う列」と「使わない列」に分けます。

In [1]:
import pandas as pd

# 学習用・予測用のデータを読み込む
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# 列名の一覧を表示する
print(list(train.columns))

['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked', 'Boat', 'Body', 'HomeDest']


| 列 | 扱い | 理由 |
|---|---|---|
| `Survived` | 予測の対象(正解) | 特徴量には入れない |
| `PassengerId` | 使わない(提出ファイルで使う) | 乗客を区別する番号で、生存とは関係しない |
| `Boat`・`Body`・`HomeDest` | 使わない | 予測用のデータにない。`Boat`・`Body` は事故の後の情報(リーク) |
| `Ticket` | 今回は使わない | 番号の形式がばらばらで、そのままでは使いにくい |
| `Name` | 加工して使う | 敬称を取り出して特徴量にする(単元4) |
| `Cabin` | 加工して使う | 記録があるかどうかを特徴量にする(単元2) |
| `Pclass`・`Sex`・`Age`・`SibSp`・`Parch`・`Fare`・`Embarked` | 使う | 必要に応じて補完・数値化する |

## ルール1:学習用と予測用に同じ処理をする

モデルは、学習用のデータで学んだ「特徴量の意味」を前提に予測します。例えば学習用で `Sex` を「female = 1、male = 0」にしたのに、予測用で逆にしてしまうと、モデルは女性を男性として扱ってしまいます。**前処理は、学習用と予測用のデータにまったく同じ方法で行います。**

## ルール2:補完などに使う値は学習用のデータから求める

欠損値を中央値で埋めるときなどに使う「中央値」は、**学習用のデータだけ** から求め、その値を予測用のデータにも使います。

本番の予測では、予測するデータは前もって手元にないのが普通です。予測用のデータの情報を使って前処理をすると、手元にあるはずのない情報がモデルづくりに混ざってしまいます(リークの一種)。また、学習用と予測用で別々の中央値を使うと、ルール1(同じ処理)も守れません。

学習用と予測用で、年齢の中央値を比べてみましょう。

In [2]:
# 年齢の中央値を、学習用・予測用それぞれから求める(欠損値は除いて計算される)
print("train:", train["Age"].median())
print("test :", test["Age"].median())

train: 29.0
test : 27.0


学習用の中央値は 29.0 歳、予測用は 27.0 歳で、データによって値が少しずつ違います。前処理では、学習用から求めた 29.0 歳を両方に使います。

## 演習

学習用のデータから、予測に使わない列(`Boat`・`Body`・`HomeDest`・`Ticket`)を取り除いた表を作り、残った列名をリストにして表示してください。

In [ ]:
# @starter
import pandas as pd

# 学習用のデータを読み込む
train = pd.read_csv("train.csv")

# 予測に使わない列の名前のリスト
unused = ["Boat", "Body", "HomeDest", "Ticket"]

# 列を取り除いた表を作る(____ に、取り除く列を指定する引数の名前を入れる)
train_used = train.drop(____=unused)

# 残った列名をリストにして表示する
print(list(train_used.columns))

In [3]:
# @solution
# @title: 使わない列を取り除く
# @hint: drop(columns=列名のリスト) で、指定した列を取り除いた表が作れます。
# @explanation: drop(columns=...) で列を取り除きます。残った 11 列のうち、Survived は正解、PassengerId は提出ファイル用なので、特徴量の材料になるのは残りの 9 列です。
import pandas as pd

# 学習用のデータを読み込む
train = pd.read_csv("train.csv")

# 予測に使わない列の名前のリスト
unused = ["Boat", "Body", "HomeDest", "Ticket"]

# columns= で指定した列を取り除いた表を作る(元の train は変わらない)
train_used = train.drop(columns=unused)

# 残った列名をリストにして表示する
print(list(train_used.columns))

['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Cabin', 'Embarked']


運賃 `Fare` の中央値を、学習用のデータから求めてください。その値で、学習用・予測用それぞれの `Fare` の欠損値を埋め、埋めた後の欠損値の数を表示してください(学習用、予測用の順に1行ずつ)。

In [ ]:
# @starter
import pandas as pd

# 学習用・予測用のデータを読み込む
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# 運賃の中央値を求める(____ に、どちらのデータから求めるかを考えて変数名を入れる)
fare_median = ____["Fare"].median()

# fillna で、欠損値を中央値で埋める(学習用・予測用の両方に同じ値を使う)
train["Fare"] = train["Fare"].fillna(fare_median)
test["Fare"] = test["Fare"].fillna(fare_median)

# 埋めた後の欠損値の数を表示する
print(train["Fare"].isna().sum())
print(test["Fare"].isna().sum())

In [4]:
# @solution
# @title: 学習用の中央値で両方を埋める
# @hint: 補完に使う値は、学習用のデータ(train)から求めます。
# @explanation: 中央値は学習用のデータ(train)から求め、その値で学習用・予測用の両方を埋めます。どちらも欠損値は 0 になります。
import pandas as pd

# 学習用・予測用のデータを読み込む
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# 運賃の中央値を、学習用のデータから求める
fare_median = train["Fare"].median()

# fillna で、欠損値を中央値で埋める(学習用・予測用の両方に同じ値を使う)
train["Fare"] = train["Fare"].fillna(fare_median)
test["Fare"] = test["Fare"].fillna(fare_median)

# 埋めた後の欠損値の数を表示する
print(train["Fare"].isna().sum())
print(test["Fare"].isna().sum())

0
0


<!-- @question
type: choice
answer: 学習用のデータから求めた値を、学習用と予測用の両方に使う。
title: 補完に使う値の求め方
hint: 本番では、予測するデータは前もって手元にありません。また、学習用と予測用には同じ処理をする必要があります。
explanation: 補完に使う値は学習用のデータだけから求め、その値を学習用と予測用の両方に使います。予測用のデータから求めると、本番では手に入らない情報を使うことになり(リーク)、別々の値を使うと学習用と予測用で処理が変わってしまいます。
-->
年齢の欠損値を中央値で埋めるとき、中央値の求め方として最も適切なものを選べ。

- 学習用のデータから求めた値を、学習用と予測用の両方に使う。
- 学習用は学習用の中央値、予測用は予測用の中央値で、それぞれ埋める。
- 学習用と予測用をつなげた全データから求めた値を使う。
- 予測用のデータから求めた値を、学習用と予測用の両方に使う。

## まとめ

- モデルに入れるデータは、すべて数値で欠損がない必要がある。そのための作業が前処理。
- 予測用のデータにない列や、事故の後の情報(リーク)は使わない。
- ルール1:学習用と予測用に同じ前処理をする。ルール2:補完などに使う値は学習用のデータだけから求める。